# Kailia's voice: make every clip (free GPU)

1. At the top: **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all.** If Colab asks to restart after installing, click **Restart session**, then **Run all** again.
3. It takes about 10 minutes. At the end a file called **kailia-voice.zip** downloads.
4. Send that zip to Claude in the project thread. Claude puts the clips in the app.

Everything runs in your own Colab. Nothing is sent to any voice company.

In [ ]:
# 1. Check the GPU and install the free voice tools (Chatterbox, Kokoro)
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!pip -q install chatterbox-tts kokoro-onnx soundfile "setuptools<81"
!ffmpeg -version | head -1

In [ ]:
# 2. The voice script (a copy of scripts/generate_voices.py) and the lines
import os
os.makedirs('/content/kailia/scripts/voice-ref', exist_ok=True)
os.chdir('/content/kailia')
open('scripts/generate_voices.py', 'w').write('#!/usr/bin/env python3\n"""\ngenerate_voices.py: make every voice clip the app plays, offline and free.\n\nKailia\'s voice (the main character, the narrator of the story)\n  1. A short reference clip (8-12 s) is made with Kokoro-82M (kokoro-onnx,\n     Apache-2.0): voices af_bella 0.5 + af_sky 0.5 blended, speed 1.2,\n     pitched up 3 semitones, then joined with af_heart (also pitched up).\n     It is saved to scripts/voice-ref/kailia-ref.wav and only remade when\n     its settings change.\n  2. Every Kailia line is spoken by Chatterbox (resemble-ai/chatterbox, MIT)\n     copying that reference clip: exaggeration 0.85, cfg_weight 0.3,\n     temperature 0.8 (exaggeration 1.0 for lines tagged "excited"), then\n     pitched up 1 semitone.\n\nOther characters (only the ones switched on in VOICES below) are spoken by\nKokoro directly.\n\nEvery clip then goes through the same ffmpeg chain: high-pass 80 Hz, treble\n+4.5 dB at 3.5 kHz, light compression, two-pass loudnorm to -16 LUFS /\n-1.5 dB true peak, mono 24 kHz, 96 kbps MP3.\n\nClips are saved to public/audio/<speaker>/<words>-<hash>.mp3. The hash covers\nthe text and the voice settings, so an unchanged line is skipped and a changed\none is remade. public/audio/manifest.json is rebuilt at the end:\n  { "kailia": { "<voiceKey of the sentence>": "kailia/<file>.mp3" }, ... }\nlib/voice.ts plays from that manifest and falls back to the device\'s own\nvoice for any sentence with no clip.\n\nWhere the lines come from\n  - Kailia\'s story lines: every { who: \'kailia\', text: "..." } in\n    app/story/page.tsx, one clip per sentence (split the same way as the app).\n  - Kailia\'s cheers: voice/kailia-lines.json.\n  - Extra lines for any speaker: voice/extra-lines.json (optional), a list of\n    { "who": "kailia", "text": "...", "tags": ["excited"] }.\n\nRun it\n  Chatterbox needs a GPU to be quick (a laptop CPU takes minutes per line):\n  use the Colab notebook scripts/generate_voices.ipynb (free GPU). Or locally:\n    python3 -m venv .venv && . .venv/bin/activate\n    pip install kokoro-onnx soundfile numpy chatterbox-tts torchaudio\n    python3 scripts/generate_voices.py                 # everything that changed\n    python3 scripts/generate_voices.py --only kailia   # one speaker\n    python3 scripts/generate_voices.py --ref-only      # just Kailia\'s reference clip (Kokoro, no GPU)\n    python3 scripts/generate_voices.py --dry-run       # list what would be made\n  ffmpeg must be installed (Colab has it; macOS: brew install ffmpeg).\n\nPrivacy: everything runs on your own machine (or your own Colab). The app only\never plays the finished MP3 files; it never calls a voice service.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport urllib.request\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent.parent\nOUT = ROOT / \'public\' / \'audio\'\nREF_DIR = ROOT / \'scripts\' / \'voice-ref\'\nMODEL_DIR = Path(os.environ.get(\'KOKORO_DIR\', Path.home() / \'.cache\' / \'kokoro-onnx\'))\nMODEL_URLS = {\n    \'kokoro-v1.0.onnx\': \'https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.1/kokoro-v1.0.onnx\',\n    \'voices-v1.0.bin\': \'https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.1/voices-v1.0.bin\',\n}\n\n# ─────────────────────────────────────────────────────────────────────────────\n# THE VOICE MAP: change voices here.\n#   engine \'chatterbox\': copies the reference clip (Kailia)\n#   engine \'kokoro\':     a Kokoro voice name, or a blend {name: weight}\n#   pitch:               semitones up (+) or down (-) after speaking\n#   enabled: False       keeps that speaker\'s current recordings untouched\n# Noel and the pet friends already have recorded voices Kelvin picked by ear\n# (public/voice/, scripts/record-voice.py). The reviewer\'s Kokoro picks for\n# them are listed below, switched off; set enabled to True to use them.\n# ─────────────────────────────────────────────────────────────────────────────\nVOICES = {\n    \'kailia\': {\n        \'engine\': \'chatterbox\',\n        \'reference\': {\n            # part 1: bella + sky blended, quick, pitched up 3 semitones\n            \'blend\': {\'af_bella\': 0.5, \'af_sky\': 0.5}, \'speed\': 1.2, \'pitch\': 3,\n            # part 2: af_heart, pitched up, joined on the end\n            \'join\': {\'voice\': \'af_heart\', \'speed\': 1.1, \'pitch\': 3},\n            \'text\': ("Hi! I\'m Kailia! I love adventures with my best friend Noel. "\n                     "Look at the fireflies! Let\'s count them, one, two, three!"),\n            \'join_text\': "Wow, so much fun! Let\'s go again!",\n        },\n        \'exaggeration\': 0.85, \'excited_exaggeration\': 1.0,\n        \'cfg_weight\': 0.3, \'temperature\': 0.8,\n        \'pitch\': 1,\n        \'enabled\': True,\n    },\n    \'noel\': {\'engine\': \'kokoro\', \'voice\': \'am_michael\', \'speed\': 0.9, \'pitch\': 0, \'enabled\': False},\n    \'flick\': {\'engine\': \'kokoro\', \'voice\': \'af_sky\', \'speed\': 1.15, \'pitch\': 0, \'enabled\': False},\n}\n\n# How names are spelled FOR THE VOICE ONLY (the screen always shows the real\n# spelling). Kelvin 2026-11-20: Kailia = "KAI-LEE-yah" ("Kai" rhymes with sky,\n# then "Lee-ya"); Noel = "No-EL", two beats. If a voice still says it wrong,\n# change the spelling here and run again.\nSAY_AS = {\n    \'Kailia\': \'Kye-LEE-yah\',\n    \'Noel\': \'No-ell\',\n}\n\n# the ffmpeg chain every clip goes through (before the loudness pass)\nFILTERS = \'highpass=f=80,treble=g=4.5:f=3500,acompressor=threshold=-20dB:ratio=2.5:attack=8:release=120:makeup=2\'\nLOUDNESS = {\'I\': -16, \'TP\': -1.5, \'LRA\': 11}\n\n# ── text helpers (must match lib/voice.ts) ───────────────────────────────────\n\ndef voice_key(sentence: str) -> str:\n    """Same as voiceKey() in lib/voice.ts."""\n    s = sentence.lower().replace(\'‘\', "\'").replace(\'’\', "\'")\n    return re.sub(r"[^a-z0-9\']+", \' \', s).strip()\n\n\ndef split_sentences(text: str) -> list[str]:\n    """Same as sentences() in lib/voice.ts."""\n    clean = re.sub(r\'[\\U0001F300-\\U0001FAFF☀-➿️\u200d]\', \'\', text)\n    clean = re.sub(r\'[*_~]\', \'\', clean)\n    clean = re.sub(r\'\\s+\', \' \', clean).strip()\n    parts = [p.strip() for p in re.split(r\'(?<=[.!?…])\\s+(?=[A-Z0-9"\\\'“‘])\', clean) if p.strip()]\n    return parts or [clean]\n\n\ndef tts_text(text: str) -> str:\n    for name, sound in SAY_AS.items():\n        text = re.sub(rf\'\\b{name}\\b\', sound, text)\n    return text\n\n\ndef slug(text: str) -> str:\n    return re.sub(r\'[^a-z0-9]+\', \'-\', text.lower()).strip(\'-\')[:40] or \'line\'\n\n# ── gathering the lines ──────────────────────────────────────────────────────\n\nLINES_FILE: str | None = None\n\n\ndef gather_lines(split: bool = True) -> list[dict]:\n    lines: list[dict] = []\n    if LINES_FILE:  # a ready list (the Colab notebook brings its own copy)\n        lines = json.loads(Path(LINES_FILE).read_text(encoding=\'utf-8\'))\n        return explode(lines) if split else lines\n    story_file = ROOT / \'app\' / \'story\' / \'page.tsx\'\n    story = story_file.read_text(encoding=\'utf-8\') if story_file.exists() else \'\'\n    for m in re.finditer(r"""\\{\\s*who:\\s*\'(\\w+)\',\\s*text:\\s*("(?:[^"\\\\]|\\\\.)*"|\'(?:[^\'\\\\]|\\\\.)*\')\\s*\\}""", story):\n        who, raw = m.group(1), m.group(2)\n        text = bytes(raw[1:-1], \'utf-8\').decode(\'unicode_escape\') if \'\\\\\' in raw else raw[1:-1]\n        lines.append({\'who\': who, \'text\': text, \'tags\': [\'excited\'] if \'!!\' in text else []})\n    cheers = ROOT / \'voice\' / \'kailia-lines.json\'\n    if cheers.exists():\n        for t in json.loads(cheers.read_text(encoding=\'utf-8\')):\n            lines.append({\'who\': \'kailia\', \'text\': t, \'tags\': [\'excited\']})\n    extra = ROOT / \'voice\' / \'extra-lines.json\'\n    if extra.exists():\n        lines += json.loads(extra.read_text(encoding=\'utf-8\'))\n    return explode(lines) if split else lines\n\n\ndef explode(lines: list[dict]) -> list[dict]:\n    """One clip per sentence, exactly as the app splits them."""\n    out, seen = [], set()\n    for line in lines:\n        for s in split_sentences(line[\'text\']):\n            k = (line[\'who\'], voice_key(s))\n            if not k[1] or k in seen:\n                continue\n            seen.add(k)\n            excited = \'excited\' in line.get(\'tags\', []) or s.endswith(\'!!\')\n            out.append({\'who\': line[\'who\'], \'text\': s, \'excited\': excited})\n    return out\n\n# ── audio helpers ────────────────────────────────────────────────────────────\n\ndef need(cmd: str):\n    if not shutil.which(cmd):\n        sys.exit(f\'{cmd} is not installed (needed for the audio clean-up).\')\n\n\ndef ffmpeg(*args: str) -> str:\n    r = subprocess.run([\'ffmpeg\', \'-hide_banner\', \'-nostdin\', \'-y\', *args], capture_output=True, text=True)\n    if r.returncode != 0:\n        raise RuntimeError(r.stderr[-1500:])\n    return r.stderr\n\n\ndef pitch_filter(semitones: float, rate: int) -> str:\n    """Pitch up/down by semitones while keeping the speed the same."""\n    if not semitones:\n        return \'anull\'\n    f = 2 ** (semitones / 12)\n    return f\'asetrate={int(rate * f)},aresample={rate},atempo={1 / f:.6f}\'\n\n\ndef finish(wav_in: Path, mp3_out: Path, rate: int, semitones: float = 0):\n    """Pitch, clean-up chain, two-pass loudnorm, mono 24 kHz 96 kbps MP3."""\n    pre = f\'{pitch_filter(semitones, rate)},{FILTERS}\'\n    ln = f"loudnorm=I={LOUDNESS[\'I\']}:TP={LOUDNESS[\'TP\']}:LRA={LOUDNESS[\'LRA\']}"\n    log = ffmpeg(\'-i\', str(wav_in), \'-af\', f\'{pre},{ln}:print_format=json\', \'-f\', \'null\', \'-\')\n    measured = json.loads(log[log.rindex(\'{\'):log.rindex(\'}\') + 1])\n    mp3_out.parent.mkdir(parents=True, exist_ok=True)\n    if \'inf\' in str(measured[\'input_i\']):\n        # too short for a loudness reading (under 0.4 s, e.g. "Ta-da!"):\n        # bring its loudest point up to the same true-peak limit instead\n        vol = ffmpeg(\'-i\', str(wav_in), \'-af\', f\'{pre},volumedetect\', \'-f\', \'null\', \'-\')\n        peak = float(vol.split(\'max_volume:\')[1].split(\'dB\')[0])\n        gain = LOUDNESS[\'TP\'] - peak\n        ffmpeg(\'-i\', str(wav_in), \'-af\', f\'{pre},volume={gain:.2f}dB\', \'-ac\', \'1\', \'-ar\', \'24000\',\n               \'-c:a\', \'libmp3lame\', \'-b:a\', \'96k\', str(mp3_out))\n        return\n    second = (f"{ln}:measured_I={measured[\'input_i\']}:measured_TP={measured[\'input_tp\']}"\n              f":measured_LRA={measured[\'input_lra\']}:measured_thresh={measured[\'input_thresh\']}"\n              f":offset={measured[\'target_offset\']}")\n    mp3_out.parent.mkdir(parents=True, exist_ok=True)\n    ffmpeg(\'-i\', str(wav_in), \'-af\', f\'{pre},{second}\', \'-ac\', \'1\', \'-ar\', \'24000\',\n           \'-c:a\', \'libmp3lame\', \'-b:a\', \'96k\', str(mp3_out))\n\n# ── Kokoro ───────────────────────────────────────────────────────────────────\n\n_kokoro = None\n\ndef kokoro():\n    global _kokoro\n    if _kokoro is None:\n        from kokoro_onnx import Kokoro  # pip install kokoro-onnx\n        MODEL_DIR.mkdir(parents=True, exist_ok=True)\n        for name, url in MODEL_URLS.items():\n            dest = MODEL_DIR / name\n            if not dest.exists():\n                print(f\'  downloading {name} ...\')\n                urllib.request.urlretrieve(url, dest)\n        _kokoro = Kokoro(str(MODEL_DIR / \'kokoro-v1.0.onnx\'), str(MODEL_DIR / \'voices-v1.0.bin\'))\n    return _kokoro\n\n\ndef kokoro_style(voice):\n    """A voice name, or a blend {name: weight}."""\n    k = kokoro()\n    if isinstance(voice, str):\n        return voice\n    return sum(k.get_voice_style(name) * w for name, w in voice.items())\n\n\n# The two names as sounds, the same ones Noel\'s recordings use\n# (scripts/record-voice.py KAILIA_SOUND / NOEL_SOUND, approved by Kelvin\n# 2026-11-20; scripts/name-sounds.test.mjs guards them). Kokoro is given these\n# exact sounds. Chatterbox can\'t take sounds, so a Kailia line with a name in\n# it is said by Kokoro with these sounds and then turned into Kailia\'s voice\n# (Chatterbox voice conversion): the name comes out exactly right.\nKAILIA_SOUND = \'kˌaɪlˈiːjɐ\'\nNOEL_SOUND = \'noʊˈɛl\'\nNAME_SOUNDS = [\n    (re.compile(r\'k[ˈˌ]?aɪl[ˈˌ]?i[ː]?ə\'), KAILIA_SOUND),\n    (re.compile(r\'k[ˈˌ]?eɪl[ˈˌ]?i[ː]?ə\'), KAILIA_SOUND),\n    (re.compile(r\'n[ˈˌ]?oʊl\'), NOEL_SOUND),\n]\nHAS_NAME = re.compile(r\'\\b(kailia|noel)\', re.I)\n\n\ndef name_phonemes(text: str) -> str:\n    p = kokoro().tokenizer.phonemize(text, \'en-us\')\n    for wrong, right in NAME_SOUNDS:\n        p = wrong.sub(right, p)\n    if re.search(r\'\\bkailia\', text, re.I):\n        assert KAILIA_SOUND in p, f\'Kailia sound not set: {p}\'\n    if re.search(r\'\\bnoel\', text, re.I):\n        assert NOEL_SOUND in p, f\'Noel sound not set: {p}\'\n    return p\n\n\ndef kokoro_wav(text: str, voice, speed: float, path: Path) -> int:\n    import soundfile as sf\n    samples, rate = kokoro().create(name_phonemes(text), voice=kokoro_style(voice), speed=speed, is_phonemes=True)\n    sf.write(str(path), samples, rate)\n    return rate\n\n# ── Kailia\'s reference clip ──────────────────────────────────────────────────\n\ndef reference_clip(cfg: dict) -> Path:\n    ref = cfg[\'reference\']\n    tag = hashlib.sha1(json.dumps(ref, sort_keys=True).encode()).hexdigest()[:8]\n    out = REF_DIR / \'kailia-ref.wav\'\n    stamp = REF_DIR / \'kailia-ref.settings\'\n    if out.exists() and stamp.exists() and stamp.read_text().strip() == tag:\n        return out\n    print(\'Making Kailia\\\'s reference clip with Kokoro ...\')\n    REF_DIR.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory() as tmp:\n        t = Path(tmp)\n        r1 = kokoro_wav(ref[\'text\'], ref[\'blend\'], ref[\'speed\'], t / \'a.wav\')\n        j = ref[\'join\']\n        r2 = kokoro_wav(ref[\'join_text\'], j[\'voice\'], j[\'speed\'], t / \'b.wav\')\n        ffmpeg(\'-i\', str(t / \'a.wav\'), \'-af\', f"{pitch_filter(ref[\'pitch\'], r1)},aresample=24000", str(t / \'a2.wav\'))\n        ffmpeg(\'-i\', str(t / \'b.wav\'), \'-af\', f"{pitch_filter(j[\'pitch\'], r2)},aresample=24000", str(t / \'b2.wav\'))\n        # a short breath between the two halves, then trim to 12 s at most\n        ffmpeg(\'-i\', str(t / \'a2.wav\'), \'-i\', str(t / \'b2.wav\'), \'-filter_complex\',\n               \'[0:a]apad=pad_dur=0.35[a];[a][1:a]concat=n=2:v=0:a=1,atrim=0:12,\'\n               \'loudnorm=I=-18:TP=-2,aresample=24000\', \'-ac\', \'1\', str(out))\n    stamp.write_text(tag)\n    secs = float(subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-show_entries\', \'format=duration\', \'-of\', \'csv=p=0\', str(out)],\n                                capture_output=True, text=True).stdout.strip() or 0)\n    print(f\'  {out.relative_to(ROOT)} ({secs:.1f} s)\')\n    return out\n\n# ── Chatterbox ───────────────────────────────────────────────────────────────\n\n_cb = None\n\ndef chatterbox():\n    global _cb\n    if _cb is None:\n        import torch\n        from chatterbox.tts import ChatterboxTTS  # pip install chatterbox-tts\n        device = \'cuda\' if torch.cuda.is_available() else (\'mps\' if torch.backends.mps.is_available() else \'cpu\')\n        print(f\'Loading Chatterbox on {device} (CPU is slow: use the Colab notebook) ...\')\n        _cb = ChatterboxTTS.from_pretrained(device=device)\n    return _cb\n\n\ndef chatterbox_wav(text: str, ref: Path, cfg: dict, excited: bool, path: Path) -> int:\n    import torchaudio as ta\n    model = chatterbox()\n    # Chatterbox now and then returns silence for a very short cheer\n    # ("Wheee!"): try again a few times before giving up on the line\n    for attempt in range(4):\n        wav = model.generate(\n            tts_text(text), audio_prompt_path=str(ref),\n            exaggeration=cfg[\'excited_exaggeration\'] if excited else cfg[\'exaggeration\'],\n            cfg_weight=cfg[\'cfg_weight\'], temperature=cfg[\'temperature\'] + 0.05 * attempt,\n        )\n        secs = wav.numel() / model.sr\n        longest = 1.6 + 0.55 * len(text.split())   # a take that rambles on is a bad take\n        if secs > 0.25 and secs < longest and float(wav.abs().max()) > 0.02:\n            break\n        print(f\'  (silent or rambling take, trying again)\')\n    else:\n        raise RuntimeError(f\'Chatterbox gave only silence for: {text}\')\n    ta.save(str(path), wav, model.sr)\n    return model.sr\n\n# Kailia\'s cheers, picked by hand (2026-11-25, Kelvin: the cheers made the\n# careful way sounded "boring, not a lot of enthusiasm"). Each was made by\n# Chatterbox saying the cheer at the start of a longer excited sentence\n# ("Yay! That was so much fun!"), six takes each; the cheer was cut out at the\n# first pause and the clearest take kept (checked with a speech-sound\n# recognizer). A file here named after a line\'s slug is used as is.\nTAKES = REF_DIR / \'kailia-takes\'\n\n_vc = None\n\ndef named_line_wav(text: str, ref: Path, cfg: dict, path: Path) -> int:\n    """A short cheer, or a line with Kailia\'s or Noel\'s name: Kokoro says it\n    (with the approved name sounds, in the reference clip\'s voice), then\n    Chatterbox turns it into Kailia\'s voice."""\n    global _vc\n    import torchaudio as ta\n    r = cfg[\'reference\']\n    with tempfile.TemporaryDirectory() as tmp:\n        t = Path(tmp)\n        rate = kokoro_wav(text, r[\'blend\'], r[\'speed\'] * 0.95, t / \'k.wav\')\n        # no pitch change before converting (it blurred the words), and a\n        # little silence each side so the first word isn\'t swallowed\n        ffmpeg(\'-i\', str(t / \'k.wav\'), \'-af\', \'adelay=300|300,apad=pad_dur=0.3,aresample=24000\', str(t / \'k2.wav\'))\n        if _vc is None:\n            from chatterbox.vc import ChatterboxVC\n            _vc = ChatterboxVC.from_pretrained(chatterbox().device)\n        import torch\n        torch.manual_seed(0)  # the same take every run (seed 0 checked: "kai-LEE-ya", "no-EL")\n        wav = _vc.generate(str(t / \'k2.wav\'), target_voice_path=str(ref))\n        ta.save(str(t / \'vc.wav\'), wav, _vc.sr)\n        # trim the padding back off\n        ffmpeg(\'-i\', str(t / \'vc.wav\'), \'-af\',\n               \'silenceremove=start_periods=1:start_threshold=-45dB,areverse,\'\n               \'silenceremove=start_periods=1:start_threshold=-45dB,areverse,adelay=60,apad=pad_dur=0.12\', str(path))\n    return _vc.sr\n\n# ── main ─────────────────────────────────────────────────────────────────────\n\ndef clip_name(who: str, line: dict, cfg: dict) -> str:\n    settings = {k: v for k, v in cfg.items() if k != \'enabled\'}\n    h = hashlib.sha1(json.dumps([line[\'text\'], line[\'excited\'], SAY_AS, FILTERS, LOUDNESS, settings, KAILIA_SOUND, NOEL_SOUND, \'vc2\' if (HAS_NAME.search(line[\'text\']) or len(line[\'text\'].split()) <= 4) else \'tts\'],\n                                sort_keys=True).encode()).hexdigest()[:10]\n    take = TAKES / f\'{slug(line["text"])}.wav\'\n    if who == \'kailia\' and take.exists():\n        h = hashlib.sha1(take.read_bytes() + h.encode()).hexdigest()[:10]\n    return f\'{who}/{slug(line["text"])}-{h}.mp3\'\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\'--only\', help=\'one speaker (e.g. kailia)\')\n    ap.add_argument(\'--ref-only\', action=\'store_true\', help="just make Kailia\'s reference clip")\n    ap.add_argument(\'--dry-run\', action=\'store_true\', help=\'list what would be made\')\n    ap.add_argument(\'--force\', action=\'store_true\', help=\'remake every clip\')\n    ap.add_argument(\'--lines\', help=\'read the lines from this JSON list instead of the app ([{who, text, tags}])\')\n    ap.add_argument(\'--print-lines\', action=\'store_true\', help=\'print every line as JSON (for the Colab notebook)\')\n    args = ap.parse_args()\n    global LINES_FILE\n    LINES_FILE = args.lines\n    if args.print_lines:\n        print(json.dumps(gather_lines(split=False), indent=1, ensure_ascii=False))\n        return\n    need(\'ffmpeg\')\n\n    if args.ref_only:\n        reference_clip(VOICES[\'kailia\'])\n        return\n\n    lines = gather_lines()\n    manifest_path = OUT / \'manifest.json\'\n    old = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}\n    manifest: dict[str, dict[str, str]] = {}\n    todo = []\n    for line in lines:\n        who = line[\'who\']\n        cfg = VOICES.get(who)\n        if not cfg or not cfg.get(\'enabled\'):\n            continue\n        name = clip_name(who, line, cfg)\n        manifest.setdefault(who, {})[voice_key(line[\'text\'])] = name\n        if args.only and who != args.only:\n            if voice_key(line[\'text\']) in old.get(who, {}):\n                manifest[who][voice_key(line[\'text\'])] = old[who][voice_key(line[\'text\'])]\n            continue\n        if args.force or not (OUT / name).exists():\n            todo.append((who, line, cfg, name))\n\n    print(f\'{len(lines)} sentences, {len(todo)} to make, {sum(map(len, manifest.values())) - len(todo)} unchanged.\')\n    if args.dry_run:\n        for who, line, _, name in todo:\n            print(f\'  {who:8} {"(excited) " if line["excited"] else ""}{line["text"]}  ->  {name}\')\n        return\n\n    ref = reference_clip(VOICES[\'kailia\']) if any(t[0] == \'kailia\' for t in todo) else None\n    for i, (who, line, cfg, name) in enumerate(todo, 1):\n        print(f\'[{i}/{len(todo)}] {who}: {line["text"]}\')\n        with tempfile.TemporaryDirectory() as tmp:\n            raw = Path(tmp) / \'raw.wav\'\n            # Chatterbox makes up sounds on very short lines ("Yay!" came out as\n            # babble, checked 2026-11-25), and can\'t take the name sounds: those\n            # lines are said by Kokoro and turned into Kailia\'s voice instead\n            take = TAKES / f"{slug(line[\'text\'])}.wav"\n            if who == \'kailia\' and take.exists():\n                # a picked take (see TAKES): straight into the clean-up chain\n                import soundfile as sf\n                rate = sf.info(str(take)).samplerate\n                shutil.copy(take, raw)\n            elif cfg[\'engine\'] == \'chatterbox\' and (HAS_NAME.search(line[\'text\']) or len(line[\'text\'].split()) <= 4):\n                rate = named_line_wav(line[\'text\'], ref, cfg, raw)\n            elif cfg[\'engine\'] == \'chatterbox\':\n                rate = chatterbox_wav(line[\'text\'], ref, cfg, line[\'excited\'], raw)\n            else:\n                rate = kokoro_wav(line[\'text\'], cfg[\'voice\'], cfg.get(\'speed\', 1.0), raw)\n            finish(raw, OUT / name, rate, cfg.get(\'pitch\', 0))\n\n    # remove clips no line uses any more, then save the manifest\n    keep = {f for m in manifest.values() for f in m.values()}\n    for f in OUT.glob(\'*/*.mp3\'):\n        if f\'{f.parent.name}/{f.name}\' not in keep and f.parent.name in manifest:\n            f.unlink()\n    OUT.mkdir(parents=True, exist_ok=True)\n    manifest_path.write_text(json.dumps(manifest, indent=1, sort_keys=True) + \'\\n\')\n    print(f\'Saved {manifest_path.relative_to(ROOT)}\')\n\n\nif __name__ == \'__main__\':\n    main()\n')
open('lines.json', 'w').write('[\n {\n  "who": "kailia",\n  "text": "Hi!! I\'m Kailia! I love to explore! In my world, letters sing and numbers dance!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "And this is my best friend Noel! He\'s a big, gentle panda.",\n  "tags": []\n },\n {\n  "who": "noel",\n  "text": "Let\'s try together!",\n  "tags": []\n },\n {\n  "who": "kailia",\n  "text": "He always says that!",\n  "tags": []\n },\n {\n  "who": "kailia",\n  "text": "Look! We found a magic map! It has six lands: a firefly forest, a dragon\'s cave, talking signposts, a story village, echo caves and a wise owl\'s tower!",\n  "tags": []\n },\n {\n  "who": "kailia",\n  "text": "Will you come with us? There\'s no wrong way to explore. Let\'s go!!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Yay!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "I did it!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Wheee!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Woohoo!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Hooray!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "We did it!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Yippee!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "So fun!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Look at me!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "I can do it!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Ta-da!",\n  "tags": [\n   "excited"\n  ]\n },\n {\n  "who": "kailia",\n  "text": "Again, again!",\n  "tags": [\n   "excited"\n  ]\n }\n]\n')
print('ready')

In [ ]:
# 3. Kailia's reference clip (made with Kokoro; the same one as in the app's repository)
import base64
ref = '...'
if ref:
    open('scripts/voice-ref/kailia-ref.wav', 'wb').write(base64.b64decode(ref))
    open('scripts/voice-ref/kailia-ref.settings', 'w').write('84eeeeea')
print('reference clip', 'ready' if ref else 'will be made now')

In [ ]:
# 4. Make every Kailia line with Chatterbox (about 10 minutes on a T4)
!python3 scripts/generate_voices.py --lines lines.json --only kailia

In [ ]:
# 5. Download the clips: send kailia-voice.zip to Claude in the project thread
!cd public && zip -qr ../kailia-voice.zip audio
from google.colab import files
files.download('kailia-voice.zip')

**Want to hear one first?** Run this cell after step 4.

In [ ]:
import glob, IPython.display as d
for f in sorted(glob.glob('public/audio/kailia/*.mp3'))[:5]:
    print(f); d.display(d.Audio(f))